In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("EnergyETL") \
    .getOrCreate()

# Device data
device_data = [
    (1, "Air Conditioner", "Living Room", 5.5),
    (2, "Refrigerator", "Kitchen", 2.1),
    (3, "LED TV", "Living Room", 1.2),
    (4, "Fan", "Bedroom", 0.8)
]

columns = [
    "device_id",
    "device_name",
    "room_name",
    "energy_kwh"
]

df = spark.createDataFrame(
    device_data,
    columns
)

# Daily summary
summary_df = df.groupBy(
    "room_name"
).agg(
    sum("energy_kwh").alias("total_energy")
)

# Add alerts
summary_df = summary_df.withColumn(
    "alert",
    when(
        col("total_energy") > 5,
        "High Usage"
    ).otherwise("Normal")
)

print("Energy Summary:")
summary_df.show()

# Save as Delta Table
summary_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("smart_home_energy_summary")

print("Delta table created.")

Energy Summary:
+-----------+------------+----------+
|  room_name|total_energy|     alert|
+-----------+------------+----------+
|Living Room|         6.7|High Usage|
|    Kitchen|         2.1|    Normal|
|    Bedroom|         0.8|    Normal|
+-----------+------------+----------+

Delta table created.
